# Standard and custom line profiles

Use a ready-made line set for Gaussian, Lorentzian or Voigt lines. For a custom shape, `LineSet` handles transition positions, template membership and weights while your function defines the profile. Coordinates below are numeric Angstrom and flux densities are in arbitrary units.

A line table lists physical transitions; repeated names group them into templates with one scale parameter. Prism supplies AGN line lists for a quick first setup, or you can pass your own table.

## Line measurement contract

Import the measurement helpers from `prism.modeling.lines.measurements`.

| Entry point | Scope |
| --- | --- |
| `lines(result, component="narrow")` | Every transition in **one** line component, measured separately. Omit `component` only for a standalone `LineSet` or its single-component fit. |
| `select_line(result, "Hb", components=["narrow", "broad"])` | One identified transition, optionally summed across kinematic components. An explicit `position` can also select members of a named template. |

Both provide `.flux()` and `.eqw(continuum=...)`: the component view returns tables; the selected profile returns individual measurements. A bare model supplies nominal values; a `FitResult` retains available covariance and links; a `BootstrapResult` supplies correlated refit samples. Methods compute when called; keep their returned values or tables for reuse.

For a selected profile, `.sample(continuum="host")` propagates the same parameter draws into profile metrics and EW. Without `continuum`, it computes profile metrics only. Covariance draws require uncertainty information; bootstrap inputs reuse existing refits rather than drawing new parameters. With a continuum **name**, its fitted uncertainty and correlations are included; an external number or callable is treated as exact.

See [Model Single Spectrum](Model%20Single%20Spectrum.ipynb) for the roles of data, models, fitters and results.


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u

from prism.modeling import Component, LineSet, select_line
from prism.modeling.lines import GaussianLines, LorentzianLines, VoigtLines
from prism.modeling.fitting import TRF

## Ready-made line sets

Rows with the same name form a template. The two OIII transitions share an amplitude, multiplied by each row's weight. Built-in kinematics use km/s; amplitudes are profile peaks, so a peak ratio need not equal an integrated-flux ratio when wavelength widths differ.

Table labels keep scientific notation, while parameter names are cleaned for Python: `[O III]5007` becomes `amp_oiii5007`. Known built-in velocity units appear in the model display; unspecified amplitude units stay blank.

`lines` accepts a preset name, an explicit file path, a list of names/paths, an Astropy `Table`/`QTable`, or a dictionary of named arrays as below. Positions carry spectral units; `weight` is optional and defaults to one. Bare position arrays need names and units before they can define a line table.

In [ ]:
linetable = {"name": ["Hb", "OIII", "OIII"],
             "position": [4861.3, 4958.9, 5006.8]*u.AA,
             "weight": [1.0, 1.0/2.98, 1.0]}

In [ ]:
ready = GaussianLines(linetable, amplitude=2., fwhm=300., instfwhm=120., name="ready", normalization='amplitude')
lorentzian = LorentzianLines(linetable, fwhm=300., instfwhm=120., name="lorentzian", normalization='amplitude')
voigt = VoigtLines(linetable, fwhm_G=180., fwhm_L=240., instfwhm=120., name="voigt", normalization='amplitude')
ready

In [ ]:
from prism.modeling.lines import measurements
measurements.lines(ready).flux()  # Total analytic integral per transition, before any response.

## Define a profile and choose shared parameters

Now define a custom profile. This Gaussian shares `sigma` in Angstrom to demonstrate the interface; a shared velocity width is usually the more physical kinematic constraint. `parameters` defines independent values per named template; `scale` names the argument multiplied by transition weights, without implicit normalization. `position` names the argument receiving the transition coordinate.

In [ ]:
from prism.modeling.lines.profiles import gaussian, gaussian_deriv


def derivative(x, amplitude, center, sigma):
    _, d_amplitude, d_center, d_sigma = gaussian_deriv(x, amplitude, center, sigma)
    return np.array([d_amplitude, d_center, d_sigma])


narrow = LineSet(
    gaussian, linetable,
    shared={"sigma": 2.0},
    parameters={"Hb": {"amplitude": 2.0}, "OIII": {"amplitude": 3.0}},
    scale="amplitude", position="center", derivative=derivative, name="narrow")
narrow.sigma.bounds = (0.3, 8.0)
{name: parameter.value for name, parameter in narrow.params.items()}

## Fit with a continuum

With the same Gaussian sigma in Angstrom for both transitions, the OIII integrated-flux ratio equals its peak ratio. Generate synthetic data with different starting and true values.

In [ ]:
def continuum(x, amplitude, slope):
    return amplitude + slope*(x-4950.0)/100.0


host = Component(continuum, {"amplitude": 1.0, "slope": 0.0}, name="host")
model = host + narrow

In [ ]:
x = np.linspace(4800, 5060, 1600)
truth = narrow.copy()
truth.sigma = 1.4
truth.amp_hb = 4.0
truth.amp_oiii = 7.0
expected = continuum(x, 1.3, 0.15) + truth(x)
yerr = np.full(x.size, 0.03)
y = expected + np.random.default_rng(41).normal(0, yerr)

The fit returns a `FitResult`: fitted model, data, parameter estimates and covariance. See [Custom Components](Custom%20Components.ipynb) for the minimal fitting workflow.

In [ ]:
fitter = TRF()
result = fitter(model, x, y, yerr=yerr)
fitted = result.model

result.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(x, y, color="0.7", linewidth=0.7, label="Data")
ax.plot(x, result.predict(), label="Fit")
ax.set(xlabel="Wavelength [Angstrom]", ylabel="Flux density")
ax.legend()

## Measure a transition or a multiplet

Keep the `FitResult` as the parent so parameter covariance remains available. `measure()` integrates over a finite window; covariance propagation is a local approximation, not posterior sampling. Passing the continuum name includes its fitted uncertainty and correlations in EW.

`to_frame()` returns one row per metric, with its value, uncertainty, interval and unit. Unavailable uncertainties remain missing.


In [ ]:
selected = select_line(result, "OIII", components="narrow")
measured = selected.measure()
uncertainty = selected.sample(n_samples=128, method="covariance", random_state=41,
                              continuum="host")
eqw = selected.eqw(continuum="host")
uncertainty.to_frame(metrics=["flux", "fwhm", "moment1", "eqw"])


In [ ]:
doublet = select_line(result, "OIII", components="narrow",
                      position=[4958.9, 5006.8]*u.AA)
doublet_result = doublet.measure()
{"single_transition_flux": measured.flux.value,
 "doublet_flux": doublet_result.flux.value, "equivalent_width": eqw}

In [ ]:
# A separate EW for every transition in one component.
measurements.lines(result, component="narrow").eqw(continuum="host", x=x)


## A single exponential line and a line set

This symmetric exponential is normalized by integrated flux. The same function defines a single `Component` or a `LineSet`; transition weights therefore multiply flux, not peak amplitude. Numerical derivatives are used here.


In [ ]:
def ExponentialLine(x, flux, center, width):
    return flux*np.exp(-np.abs(x-center)/width)/(2*width)


single = Component(
    ExponentialLine, {"flux": 5.0, "center": 4861.3, "width": 4.0}, name="single")
single.width.bounds = (0.1, None)
single


For several transitions, share the width in Angstrom and give each template its own flux. To make all parameters independent, use `shared={}` and put `width` inside each template's `parameters` dictionary.


In [ ]:
broad = LineSet(
    ExponentialLine, linetable, shared={"width": 4.0},
    parameters={"Hb": {"flux": 5.0}, "OIII": {"flux": 8.0}},
    scale="flux", position="center", name="broad")
broad.width.bounds = (0.1, None)
broad


The same measurement methods work on a generic `LineSet` with an explicit integration grid. Automatic windows require a recognized width convention; peak-width metrics require a suitable peak.


In [ ]:
custom = select_line(broad, "Hb")
custom_measurement = custom.measure(x=x)
custom_measurement.to_frame(metrics=["flux", "fwhm"])

## Explicit shared velocity parameters

Next, put the kinematic convention inside the profile. It receives the rest position. It owns the Doppler convention, velocity-to-coordinate width conversion and peak scaling. The example below reuses Prism's scientific conversion functions: logarithmic velocity shift and the existing redshift amplitude convention. Widths are numeric km/s. Equal velocity widths imply different Angstrom widths, so a peak ratio is generally different from an integrated-flux ratio.

In [ ]:
from prism.modeling.lines.profiles import observed_center, gaussian_profile_params

def velocity_gaussian(x, amplitude, position, offset, fwhm, redshift):
    center = observed_center(position, offset, redshift)
    peak, sigma = gaussian_profile_params(center, amplitude, 1.0, fwhm, redshift, 0.0)
    return gaussian(x, peak, center, sigma)

kinematic = LineSet(
    velocity_gaussian, linetable,
    shared={"offset": -85.0, "fwhm": 280.0, "redshift": 0.0},
    parameters={"Hb": {"amplitude": 2.0}, "OIII": {"amplitude": 3.0}},
    position="position", name="kinematic")
kinematic.redshift.fixed = True
kinematic.fwhm.bounds = (20, 1500)
kinematic

## Gaussian instrumental width, separate from line grouping

Alternatively, configure a built-in profile directly. It owns its instrumental width. Scalar `instfwhm=150.0` means 150 km/s everywhere. A two-column numeric table gives observed position in Angstrom and instrumental FWHM in km/s. The spline and its derivative are prepared once, then sampled at each current observed line center. The intrinsic and instrumental Gaussian widths combine in quadrature. The analytic derivative includes the slope of the width curve.

`LorentzianProfile` and `VoigtProfile` use the same Gaussian calibration: the former becomes a Voigt; the latter broadens only its Gaussian part. Peak amplitude follows the existing Prism convention; this local width prescription does not automatically preserve integrated flux or replace a general response operator.

In [ ]:
from prism.modeling.lines import GaussianProfile

instrument_curve = np.array([[4800.0, 100.0], [5060.0, 180.0]])
gaussian_profile = GaussianProfile(instfwhm=instrument_curve, position_unit=u.AA)
calibrated = LineSet(
    gaussian_profile, linetable,
    shared={"offset": 0.0, "fwhm": 300.0, "redshift": 0.0},
    parameters={"Hb": {"amplitude": 2.0}, "OIII": {"amplitude": 3.0}},
    position="position", derivative=gaussian_profile.derivative, name="calibrated")
calibrated.redshift.fixed = True
calibrated.fwhm.bounds = (20.0, 1500.0)
calibrated

## Reuse and convert line tables

For another application, start from the supplied lists and inspect a named template. ECSV stores positions, relative weights and their coordinate convention. Energy/frequency conversion changes positions and rescales weights with the spectral Jacobian; air/vacuum conversion changes the wavelength medium.

In [ ]:
from pathlib import Path
from astropy.table import QTable
from prism.modeling.lines import (
    convert_domain_csv, convert_linetable_medium)
from prism.modeling import lines

temp_dir = Path("results/line-tables")
energy_dir = temp_dir / "energy"
lines.set_directory(temp_dir)
lines.export("oiii", directory=temp_dir, window=(4000*u.AA, 7000*u.AA), overwrite=True)

In [ ]:
air_table = lines.load("oiii")
air_table

The table keeps `[OIII]5007` as the physical template tag; its model parameter is `amp_oiii5007`. Use the original tag with `select_line`, and the cleaned name with `params` or attribute access.

In [ ]:
convert_domain_csv(temp_dir, output_unit=u.eV, output_dir=energy_dir, overwrite=True)
first_table = QTable.read(energy_dir / "oiii.ecsv", format="ascii.ecsv")
energy_lines = GaussianLines(first_table, name="energy_lines", normalization='amplitude')
first_table

In [ ]:
x_energy = np.linspace(2.4, 2.6, 1000)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(x_energy, energy_lines(x_energy))
ax.set(xlabel="Energy [eV]", ylabel="Flux density per eV")

In [ ]:
vacuum_table = convert_linetable_medium(air_table, target_medium="vacuum")
vacuum_table

Use either a response operator or `instfwhm` for the same instrumental broadening, to avoid applying it twice. Built-in profiles store their calibration; custom functions must be supplied when loading saved models.

In [ ]:
from prism.modeling.io import load_model

Path("results").mkdir(exist_ok=True)
combined = fitted + broad + kinematic + calibrated
combined.save("results/line-components.fits", overwrite=True)
restored = load_model(
    "results/line-components.fits",
    components={"host": host, "narrow": narrow, "broad": broad, "kinematic": kinematic})
restored

### Integrated flux as the fitted parameter

Ready-made Gaussian, Lorentzian and Voigt sets default to `normalization="flux"`.
`flux_<template>` is the integral of the full transition with weight one. Other
members receive this flux times their table weight, independent of width.
The packaged [O III] weights approximate 5007/4959 = 3; precision atomic ratios
can be supplied in a custom table. This is an input convention, not an inferred
ratio. Explicit `normalization="amplitude"` is also available.

In [ ]:
from prism.modeling.lines import measurements
from prism.modeling.lines import GaussianLines

flux_model = GaussianLines("oiii", flux=12., fwhm=250., name="flux_lines",
                           bounds={"flux": (0., 100.)})
print(flux_model.flux_oiii5007.value)  # Total 5007 flux in the chosen units.
print(measurements.lines(flux_model).flux())             # Per-transition integrals, including weights.
flux_model.fwhm = 500.               # Broader, lower peak; identical total flux.
measurements.lines(flux_model).flux()